## 2. GLUE dataset과 Huggingface

### Huggingface transformers 설치 및 환경 구성

In [1]:
!pip install transformers
!pip install accelerate

In [2]:
!python -c "from transformers import pipeline; print(pipeline('sentiment-analysis')('I love you'))"

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
config.json: 100%|█████████████████████████████| 629/629 [00:00<00:00, 2.24MB/s]
model.safetensors: 100%|█████████████████████| 268M/268M [00:04<00:00, 63.4MB/s]
Loading weights: 100%|██████████████████████| 104/104 [00:00<00:00, 4312.87it/s]
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 180kB/s]
vocab.txt: 232kB [00:00, 39.7MB/s]
[{'label': 'POSITIVE', 'score': 0.9998656511306763}]


## 3. 커스텀 프로젝트 제작

### (1) Datasets

### Huggingface dataset에서 불러오기

In [3]:
!pip install datasets

In [4]:
import datasets
from datasets import load_dataset

huggingface_mrpc_dataset = load_dataset('glue', 'mrpc')
print(huggingface_mrpc_dataset)

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


In [5]:
train = huggingface_mrpc_dataset['train']
cols = train.column_names
cols

['sentence1', 'sentence2', 'label', 'idx']

In [6]:
for i in range(5):
    for col in cols:
        print(col, ":", train[col][i])
    print('\n')

sentence1 : Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
sentence2 : Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .
label : 1
idx : 0


sentence1 : Yucaipa owned Dominick 's before selling the chain to Safeway in 1998 for $ 2.5 billion .
sentence2 : Yucaipa bought Dominick 's in 1995 for $ 693 million and sold it to Safeway for $ 1.8 billion in 1998 .
label : 0
idx : 1


sentence1 : They had published an advertisement on the Internet on June 10 , offering the cargo for sale , he added .
sentence2 : On June 10 , the ship 's owners had published an advertisement on the Internet , offering the explosives for sale .
label : 1
idx : 2


sentence1 : Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
sentence2 : Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
label : 0

### 커스텀 데이터셋 만들기

In [9]:
import os

import pandas as pd
from datasets import Dataset

def parse_mrpc_file(file_path):
    """MRPC 파일을 안전하게 파싱하는 함수"""
    data = {
        'Quality': [],
        '#1 ID': [],
        '#2 ID': [], 
        '#1 String': [],
        '#2 String': []
    }
    
    with open(file_path, 'r', encoding='utf-8') as f:
        # 헤더 스킵
        next(f)
        
        for line_num, line in enumerate(f, 1):
            try:
                parts = line.strip().split('\t')
                if len(parts) >= 5:
                    # 5개 컬럼으로 분할 (마지막 탭들은 모두 마지막 컬럼에 포함)
                    quality = int(parts[0])
                    id1 = int(parts[1]) 
                    id2 = int(parts[2])
                    string1 = parts[3]
                    string2 = '\t'.join(parts[4:])  # 나머지 모든 부분을 합침
                    
                    data['Quality'].append(quality)
                    data['#1 ID'].append(id1)
                    data['#2 ID'].append(id2)
                    data['#1 String'].append(string1)
                    data['#2 String'].append(string2)
                else:
                    print(f"Line {line_num}: Invalid format, skipping")
            except Exception as e:
                print(f"Line {line_num}: Error {e}, skipping")
    
    return pd.DataFrame(data)

# 로컬 파일에서 MRPC 데이터 읽기
train_df = parse_mrpc_file(os.getenv('HOME') + '/data/msr_paraphrase_train.txt')
test_df = parse_mrpc_file(os.getenv('HOME') + '/data/msr_paraphrase_test.txt')

In [10]:
# 데이터 구조 확인
print("Train dataset columns:", train_df.columns.tolist())
print("Train dataset shape:", train_df.shape)
print("\nFirst 5 examples:")

for i in range(5):
    row = train_df.iloc[i]
    for col in train_df.columns:
        print(f"{col}: {row[col]}")
    print('\n')

Train dataset columns: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String']
Train dataset shape: (4076, 5)

First 5 examples:
Quality: 1
#1 ID: 702876
#2 ID: 702977
#1 String: Amrozi accused his brother, whom he called "the witness", of deliberately distorting his evidence.
#2 String: Referring to him as only "the witness", Amrozi accused his brother of deliberately distorting his evidence.


Quality: 0
#1 ID: 2108705
#2 ID: 2108831
#1 String: Yucaipa owned Dominick's before selling the chain to Safeway in 1998 for $2.5 billion.
#2 String: Yucaipa bought Dominick's in 1995 for $693 million and sold it to Safeway for $1.8 billion in 1998.


Quality: 1
#1 ID: 1330381
#2 ID: 1330521
#1 String: They had published an advertisement on the Internet on June 10, offering the cargo for sale, he added.
#2 String: On June 10, the ship's owners had published an advertisement on the Internet, offering the explosives for sale.


Quality: 0
#1 ID: 3344667
#2 ID: 3344648
#1 String: Around 0335 GMT, 

In [11]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# train 데이터를 train(80%)과 validation(20%)으로 분할
train_indices = list(range(len(train_dataset['Quality'])))
train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42, 
                                      stratify=train_dataset['Quality'])

# validation 데이터셋 생성
validation_dataset = {}
for key in train_dataset.keys():
    validation_dataset[key] = [train_dataset[key][i] for i in val_idx]

# train 데이터셋 업데이트 (validation으로 사용된 데이터 제거)
train_dataset_final = {}
for key in train_dataset.keys():
    train_dataset_final[key] = [train_dataset[key][i] for i in train_idx]

# 허깅페이스 Dataset 객체로 변환
train_hf_dataset = Dataset.from_dict(train_dataset_final)
validation_hf_dataset = Dataset.from_dict(validation_dataset)
test_hf_dataset = Dataset.from_dict(test_dataset)

# DatasetDict 생성 (이게 허깅페이스의 표준 방식)
customized_mrpc_dataset = DatasetDict({
    'train': train_hf_dataset,
    'validation': validation_hf_dataset,
    'test': test_hf_dataset
})

# 결과 출력
print("DatasetDict({")
for split_name, split_data in customized_mrpc_dataset.items():
    print(f"    {split_name}: Dataset({{")
    print(f"        features: {list(split_data.features.keys())},")
    print(f"        num_rows: {split_data.num_rows}")
    print("    })")
print("})")

print("\n데이터셋 정보 확인!")
print(f"Train dataset shape: ({customized_mrpc_dataset['train'].num_rows}, {len(customized_mrpc_dataset['train'].features)})")
print(f"Validation dataset shape: ({customized_mrpc_dataset['validation'].num_rows}, {len(customized_mrpc_dataset['validation'].features)})")
print(f"Test dataset shape: ({customized_mrpc_dataset['test'].num_rows}, {len(customized_mrpc_dataset['test'].features)})")

print("Dataset 생성 완료!")
print(f"Train samples: {len(customized_mrpc_dataset['train'])}")
print(f"Validation samples: {len(customized_mrpc_dataset['validation'])}")
print(f"Test samples: {len(customized_mrpc_dataset['test'])}")

# 첫 번째 샘플 확인
print(f"\n첫 번째 train 샘플:")
print(customized_mrpc_dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 3260
    })
    validation: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 816
    })
    test: Dataset({
        features: ['Quality', '#1 ID', '#2 ID', '#1 String', '#2 String'],
        num_rows: 1725
    })
})

데이터셋 정보 확인!
Train dataset shape: (3260, 5)
Validation dataset shape: (816, 5)
Test dataset shape: (1725, 5)
Dataset 생성 완료!
Train samples: 3260
Validation samples: 816
Test samples: 1725

첫 번째 train 샘플:
{'Quality': 0, '#1 ID': 2121506, '#2 ID': 2121285, '#1 String': 'The festival kicked off yesterday one day after the Competition Commission delivered its final verdict to the Government on the proposed £4.1 billion merger.', '#2 String': 'The Competition Commission delivered its verdict yesterday on the proposed merger of the two big ITV players, Carlton and Granada.'}


### (2) Tokenizer와 Model

### Huggingface Auto Classes를 이용하는 방법

In [12]:
!python -m pip install "transformers[torch]" "accelerate>=1.1.0" -U

In [13]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

huggingface_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
huggingface_model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels = 2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def transform(data):
    return huggingface_tokenizer(
        data['sentence1'],
        data['sentence2'],
        truncation = True,
        padding = 'max_length',
        return_token_type_ids = False,
        )

In [15]:
hf_dataset = huggingface_mrpc_dataset.map(transform, batched=True)

# train & validation & test split
hf_train_dataset = hf_dataset['train']
hf_val_dataset = hf_dataset['validation']
hf_test_dataset = hf_dataset['test']

In [16]:
# Q. tf_train_dataset에 transform 함수를 매핑해봅시다. 어떤 오류가 발생하나요?
tf_train_dataset_error = tf_train_dataset.map(transform)

NameError: name 'tf_train_dataset' is not defined

In [17]:
# DataFrame을 dict 형식으로 변경 (각 컬럼을 리스트로 변환)
train_dataset = train_df.to_dict('list')
test_dataset = test_df.to_dict('list')

# validation set이 없는 경우, train set에서 일부를 분할
# 또는 test set을 validation으로 사용
val_dataset = test_dataset.copy()  # 예시로 test를 val로 복사

# Huggingface Dataset 생성
train_dataset = Dataset.from_dict(train_dataset)
val_dataset = Dataset.from_dict(val_dataset) 
test_dataset = Dataset.from_dict(test_dataset)

print("Dataset 생성 완료!")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# 커스텀 데이터용 transform 함수 정의
def transform_custom(batch):
    # 커스텀 파일 데이터는 이미 문자열이므로 decode 불필요
    sentence1 = batch['#1 String']
    sentence2 = batch['#2 String']
    return huggingface_tokenizer(
        sentence1,
        sentence2,
        truncation=True,
        padding='max_length',
        return_token_type_ids=False,
    )

# 토큰화 및 패딩을 적용
train_dataset = train_dataset.map(transform_custom, batched=True)
val_dataset = val_dataset.map(transform_custom, batched=True)
test_dataset = test_dataset.map(transform_custom, batched=True)

Dataset 생성 완료!
Train samples: 4076
Validation samples: 1725
Test samples: 1725


Map:   0%|          | 0/4076 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

### (3) Train/Evaluation과 Test

### Trainer를 활용한 학습

In [18]:
import os
import numpy as np
from transformers import Trainer, TrainingArguments

output_dir = 'transformers'

training_arguments = TrainingArguments(
    output_dir,                                         # output이 저장될 경로
    eval_strategy="epoch",           #evaluation하는 빈도
    learning_rate = 2e-5,                         #learning_rate
    per_device_train_batch_size = 8,   # 각 device 당 batch size
    per_device_eval_batch_size = 8,    # evaluation 시에 batch size
    num_train_epochs = 3,                     # train 시킬 총 epochs
    weight_decay = 0.01,                        # weight decay
)

In [19]:
!pip install evaluate

In [20]:
from evaluate import load
metric = load('glue', 'mrpc')

def compute_metrics(eval_pred):
    predictions,labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references = labels)

In [21]:
trainer = Trainer(
    model=huggingface_model,           # 학습시킬 model
    args=training_arguments,           # TrainingArguments을 통해 설정한 arguments
    train_dataset=hf_train_dataset,    # training dataset
    eval_dataset=hf_val_dataset,       # evaluation dataset
    compute_metrics=compute_metrics,
)
trainer.train()
print("슝~")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.388443,0.835784,0.883882
2,0.502342,0.398581,0.855392,0.899145
3,0.318031,0.541532,0.857843,0.902027


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

슝~


In [22]:
# Trainer 정의 후 실행 전 아래 한 줄을 추가하여 콜백을 강제로 제거
# from transformers.utils.notebook import NotebookProgressCallback

# trainer.remove_callback(NotebookProgressCallback)

trainer.evaluate(hf_test_dataset)

{'eval_loss': 0.6001355648040771,
 'eval_accuracy': 0.8289855072463768,
 'eval_f1': 0.8763102725366876,
 'eval_runtime': 24.7114,
 'eval_samples_per_second': 69.806,
 'eval_steps_per_second': 8.741,
 'epoch': 3.0}

In [23]:
#메모리를 비워줍니다.
del huggingface_model

In [28]:
# Q. 커스텀 데이터셋으로 학습시켜봅시다.
# huggingface_model = AutoModelForSequenceClassification.from_pretrained(
#     'distilbert-base-uncased', num_labels=2)

# trainer_custom = Trainer(
#     model=huggingface_model,
#     args=training_arguments,
#     train_dataset=tf_train_dataset,
#     eval_dataset=tf_val_dataset,
#     compute_metrics=compute_metrics,
# )
# trainer_custom.train()


# Q. 커스텀 데이터셋으로 학습시켜봅시다.

# 1) 커스텀 데이터셋에 토큰화 적용 + labels 컬럼 추가

def preprocess_custom(batch):
    # 토큰화
    tokenized = huggingface_tokenizer(
        batch["#1 String"],
        batch["#2 String"],
        truncation=True,
        padding="max_length",
        return_token_type_ids=False,
    )
    # 라벨 컬럼 이름을 Trainer 규약에 맞게 변경
    tokenized["labels"] = batch["Quality"]
    return tokenized

train_dataset_for_trainer = customized_mrpc_dataset["train"].map(
    preprocess_custom,
    batched=True,
)

val_dataset_for_trainer = customized_mrpc_dataset["validation"].map(
    preprocess_custom,
    batched=True,
)

# 2) 모델 로드
huggingface_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
)

# 3) Trainer 정의 (커스텀 토큰화된 데이터셋 사용)
trainer_custom = Trainer(
    model=huggingface_model,
    args=training_arguments,
    train_dataset=train_dataset_for_trainer,
    eval_dataset=val_dataset_for_trainer,
    compute_metrics=compute_metrics,
)

trainer_custom.train()

Map:   0%|          | 0/3260 [00:00<?, ? examples/s]

Map:   0%|          | 0/816 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.477882,0.802696,0.861803
2,0.494977,0.625786,0.806373,0.864028
3,0.297397,0.631624,0.812500,0.861538


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1224, training_loss=0.3677678170546987, metrics={'train_runtime': 510.6926, 'train_samples_per_second': 19.15, 'train_steps_per_second': 2.397, 'total_flos': 1295531158855680.0, 'train_loss': 0.3677678170546987, 'epoch': 3.0})

In [30]:
# Validation 데이터셋을 이용해 평가해봅니다.
# eval_results = trainer_custom.evaluate()

# print(eval_results)


from transformers import TrainerCallback
from transformers.utils.notebook import NotebookProgressCallback

# 콜백에서 노트북 프로그레스 콜백을 제거
callbacks = [cb for cb in trainer_custom.callback_handler.callbacks
             if not isinstance(cb, NotebookProgressCallback)]
trainer_custom.callback_handler.callbacks = callbacks

# 이제 안전하게 평가
eval_results = trainer_custom.evaluate()
print(eval_results)

{'eval_loss': 0.6316236853599548, 'eval_accuracy': 0.8125, 'eval_f1': 0.8615384615384616, 'eval_runtime': 11.9182, 'eval_samples_per_second': 68.467, 'eval_steps_per_second': 8.558, 'epoch': 3.0}


### 4. 프로젝트 : 커스텀 프로젝트 직접 만들기

In [32]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 43.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 114.1 MB/s eta 0:00:0000:01
  Attempting uninstall: protobuf━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/16 [libclang]
    Found existing installation: protobuf 5.29.3━━━━━━━━━━━━━━  1/16 [libclang]
    Uninstalling protobuf-5.29.3:m━━━━━━━━━━━━━━━━━━━━━━━━━━━  5/16 [protobuf]
      Successfully uninstalled protobuf-5.29.3━━━━━━━━━━━━━━━━  5/16 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [tensorflow]6 [tensorflow]


In [33]:
import tensorflow
import numpy
import transformers
import datasets

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)

I0000 00:00:1773285839.778306     267 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773285844.155465     267 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


2.21.0
2.2.6
5.3.0
4.7.0
